# Quil Plotting Library Tutorial

In [1]:
from quil.program import Program

# The main API is captured in the PlottableProgramPulseSchedule object:
from quil.plotting import PlottableProgramPulseSchedule

Typically a user will have their own program they want to plot. However, for the purpose of this tutorial, we provide a list of programs, and a few methods to list and load them. However, feel free to try your own here.

In [2]:
def load_program(name):
    """Parse one of the bundled test programs by name."""
    with open(f'../tests/programs/{name}.quil', 'r') as f:
        return Program.parse(f.read())

def list_programs():
    """List all of the bundled test program names."""
    import os
    for program_name in sorted(os.listdir('../tests/programs/')):
        print(program_name)

## 1. Plotting Pulse Schedules

The following one liner constructs the `PlottableProgramPulseSchedule` from our program and then renders an interactive plot in the cell's output.

In [3]:
PlottableProgramPulseSchedule(load_program('sequenced_hadamard_barrier')).draw()

alt.LayerChart(...)

### Reading the Plot

- The Y-Axis depicts **lanes** — one qubit or coupler by default — and time runs left to right along the X-Axis.
- A **pulse** is drawn as its waveform's envelope. A complex waveform containing non-zero Q elements gets a second trace for its imaginary part.
- A **frame update** — `SHIFT-PHASE`, `SET-PHASE`, `SWAP-PHASES`, etc. are marked with a symbol on their lane.
- All Quil-T Instructions are supported.


### Interactivity

The default jupyter notebook rendering is interactive.

- You can **pan** the plot by clicking and dragging, and **zoom** by scrolling.
- The legend lists the instructions plotted. You can select one (or more) by clicking (or shift-clicking) the legend items. This will dim unselected instruction types. Click on an empty space to clear your selection.
- You can **hover** over the plot elements to see their tooltip, which provides more information on the pulse.

### Changing Plot Settings

There is a lot that is customizable about the plot that goes beyond that simple opinionated one-liner. We can change how we group the pulses into lanes on the Y-axis, the color map of the instructions, and more. Let's explore this here. The `PlottableProgramPulseSchedule` follows a [builder pattern](https://en.wikipedia.org/wiki/Builder_pattern), so can change settings about it by call simply-named methods on it before the final `.draw()` call.

The following example adjusts the Y-Axis lanes to be drawn by frames rather than qubits. This change separates a qubit's charge, flux, and readout lines onto rows of their own.

In [4]:
iswap = load_program('single_gate_iswap')

schedule = PlottableProgramPulseSchedule(iswap)
schedule.with_y_axis('Frame').draw()

alt.LayerChart(...)

The options for how we can group the Y-Axis includes "Qubit" (the default one), "Frame", "Channel Type", or "Instruction":

In [5]:
schedule.with_y_axis('Instruction').draw()

alt.LayerChart(...)

For large plots, there may be a lot going on while we only care about a specific subsection. In this case, we can use the `hide()` and `show()` methods to filter what we plot. Let's look at the rotated surface code toy example.

In [6]:
PlottableProgramPulseSchedule(load_program('sequenced_1q_fenced_to_2q')).hide("ISWAP").draw()

alt.LayerChart(...)

Hide and show predicates are applied in the order they are called on the schedule. So a `show()` after a `hide()` can re-add something to the plot. Additionally, you use predicates for complex hide tasks. You can find documentation on this [here](https://rigetti.github.io/quil-rs/plotting/api/pulse-schedule.html#quil.plotting.schedule.quil.plotting.schedule.PlottableProgramPulseSchedule.PlottableProgramPulseSchedule.hide).

In [7]:
rotated = load_program('rotated-surface-code')

schedule = PlottableProgramPulseSchedule(rotated)
schedule.hide(lambda event: event.qubit != "Qubit: 10") # Hide everything that isn't Qubit 10
schedule.hide(lambda event: "measure" in event.logical_instruction_name.lower()) # Hide the measure instructions
schedule.show(lambda event: "10_readout_tx" in event.frame.name) # But show the readout_tx components
schedule.draw()

alt.LayerChart(...)

In the above example, we will notice that there are two different types of SX gates plotted: `SX_DATA` and `SX_ANCILLA_ECHO`, however with the default coloring scheme they are both the same color. We can modify how pulses are colored using the `with_color_key` and `with_color_of` methods. Internally there is a color map that maps pulses from their `color_key` to their `color`. Changing the key, changes the grouping, while calling `with_color_of` enables changing the values:

In [8]:
schedule.with_color_key("Instruction").with_color_of("SX_DATA", "#000000").draw()

alt.LayerChart(...)

The default coloring map is a bit different and doesn't compare to any of the other options. If you would like to restore that, you can set the color key to `None`.

In [9]:
schedule.with_color_key(None).draw()

alt.LayerChart(...)

There are other methods to adjust how the graph appears that are not described here. For more information, view the [PlottableProgramPulseSchedule](https://rigetti.github.io/quil-rs/plotting/) documentation.

## Plotting Schedules with Control Flow

A program with branches or loops has more than one basic block, and each is scheduled
independently — so there's no single time axis that can show them all at once.

`test_blocks` is a simple program showcasing a loop.

In [10]:
PlottableProgramPulseSchedule(load_program('test_blocks')).draw()

Passing a filename to `draw()` writes the whole thing out: the graph itself, and beside
it a `.blocks` directory holding one file per drawable block, which the graph's nodes
link into. Blocks with nothing to draw stay in the graph as plain nodes and get no file
of their own.

`with_shared_y_axis()` gives every block the same lane set, so a qubit sits at the same
height in each file.

In [11]:
from pathlib import Path

schedule = PlottableProgramPulseSchedule(load_program('test_blocks')).with_shared_y_axis()
schedule.draw('plots/test_blocks.html')

sorted(path.name for path in Path('plots/test_blocks.html.blocks').iterdir())

['block-01.rounds_start.html', 'block-03.rounds_end.html']

## Output formats

`draw(filename)` picks the format from the extension —
`html` for the interactive chart, or `svg`/`png`/`pdf` for a static one, rendered in process
without needing a browser installed.

```python
PlottableProgramPulseSchedule(program).draw('schedule.html')
PlottableProgramPulseSchedule(program).draw('schedule.svg')
```

Large programs are handled by deduplicating repeated waveform shapes, dropping runs of
constant samples, and capping the points drawn per pulse (`with_max_points_per_pulse`,
500 by default — pass `None` for exact fidelity). HTML output renders to a canvas rather
than SVG, which is what keeps panning and zooming smooth on programs with thousands of marks.

# 2. Plotting Circuits

The Quil-Plotting library can also plot Circuits, gate-level views. The method is extremely similar:

In [12]:
from quil.plotting import PlottableProgramCircuit

# Note PlottableProgramCircuit is used now
PlottableProgramCircuit(load_program('sequenced_hadamard_barrier')).draw()

alt.LayerChart(...)

Above a simple circuit is being displayed. Besides switching to `PlottableProgramCircuit` the API is largely the same. You use `.draw()` to draw the diagram and save output files, you can `.hide()`/`.show()` the same, and the diagram is configured with a series of `.with_*()` builder methods. See the [documentation](https://rigetti.github.io/quil-rs/plotting/) for more info.